<a href="https://colab.research.google.com/github/RostislavKorst/Generative-Models-2026/blob/main/Assignment%202/2_gan_dcgan_conditional.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Generative Models
***

**Autumn 2026, MSc course**


## Assignment 2

***


*Authored by Arip Asadulaev & Manh Lab*

*Adapted by Alexander Panfilov*

***


The assignment is split into three parts: at first, you will implement classical GAN for the MNIST dataset, after, you will enhance your results with DCGAN and the CIFAR dataset, then you implement conditional gan with modulation method.

Good luck!

## 1. GAN
*Source: <a href="https://arxiv.org/pdf/1406.2661.pdf" target="_blank">Goodfellow, Ian J., et al. "Generative Adversarial Networks." arXiv preprint arXiv:1406.2661 (2014)</a>.*

In 2014, Ian Goodfellow and his colleagues at the University of Montreal published a stunning paper introducing the world to GANs, or generative adversarial networks. The models play two distinct (literally, adversarial) roles. Given some real data set $\mathcal{X}$, $G$ is the generator, trying to create fake data that looks just like the genuine data, while $D$ is the discriminator, getting data from either the real set or $G$ and labeling the difference.

Goodfellow’s metaphor (and a fine one it is) was that $G$ was like a team of forgers trying to match real paintings with their output, while $D$ was the team of detectives trying to tell the difference. (Except in this case, the forgers $G$ never get to see the original data — only the judgments of $D$. They’re like blind forgers.)

In this part, we are going to implement classical GAN. You can find a rough plan on how to do it below; however, it is not mandatory to stick with it precisely but one might expect some questions related to the listed steps. Feel free to adapt provided code!

- Define the parameters
- Load the data (with transforms and normalization)
- Denormalize for visual inspection of samples
- Define the Discriminator network, the Generator network
- Study the activation function: Leaky ReLU
- Explain the output activation function: Tanh, Sigmoid
- Look at some sample outputs
- Define losses, optimizers and helper functions for training
     - For Discriminator
     - For Generator
- Train the model
- Save intermediate generated images to file
- Look at some outputs
- Save the models

### Imports

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torchvision import transforms
from torchvision.datasets import CIFAR10, MNIST
from torchvision.transforms import Compose, Normalize, ToTensor
from torchvision.utils import save_image

%matplotlib inline
plt.style.use("ggplot")

### Define parameters & data loading

In [ ]:
hidden_size = 256
batch_size = 100
LR = 0.0002
num_epochs = 300
sample_dir = "./images"
latent_size = 64  # input random input vector latent
image_size = 784  # 28*28 flatten
device = "cuda" if torch.cuda.is_available() else "cpu"

if not os.path.exists(sample_dir):
    os.makedirs(sample_dir)
## Define the dataset
mnist_dataset = MNIST(
    root="data",
    train=True,
    download=True,
    transform=Compose([ToTensor(), Normalize(mean=(0.5,), std=(0.5,))]),
)

In [ ]:
img, label = mnist_dataset[0]
print("Label: ", label)
plt.hist(img[:, 5:15, 5:15].flatten(), bins=15)

In [ ]:
# denormalization image from range (-1)-1 to range 0-1 to display it
def denorm(x):


In [ ]:
# show image sample with matplotlib

In [ ]:
# define the dataloader

### GAN implementation

GAN consists of two deep networks, a generator ($G$) and a discriminator ($D$). The generator creates images before learning it. Since the discriminator is a binary classification model, we can use the binary cross-entropy loss function to quantify how well it can distinguish between real and generated images.

In [ ]:
class Generator(nn.Module):
    def __init__(self, latent_size, image_size, hidden_size):
        super(Generator, self).__init__()
        ## set a linear layer with input size is latent_size and output size is hidden_size

        # set a linear layer with input size is hidden_size and output size is hidden_size

        # set a linear layer with input size is hidden_size and output size is image_size

    def forward(self, x):
        # forward x -> linear1 -> relu -> linear2 -> relu -> linear3 -> tanh
        return x


class Discriminator(nn.Module):
    def __init__(self, image_size, hidden_size):
        super(Discriminator, self).__init__()
        ## set a linear layer with input size is image_size and output size is hidden_size

        # set a linear layer with input size is hidden_size and output size is hidden_size

        # set a linear layer with input size is hidden_size and output size is 1

    def forward(self, x):
        # forward x -> linear1 -> relu -> linear2 -> relu -> linear3 -> sigmoid

        return x

### Create the Generator and Discriminator

In [ ]:
# create new Generator model

# G = Generator...

# create new Discriminator model

# D = Discriminator...

In [ ]:
# show the output of model
y = G(torch.randn(2, latent_size))

In [ ]:
# define the criterion is nn.BCELoss()

## Define the optimizer for generator and discrimator

Discriminator Training

In [ ]:
def reset_grad():
    ## reset gradient for optimizer of generator and discrimator

def train_discriminator(D_model, G_model, images):

    # Create the labels which are later used as input for the BCE loss
    real_labels = torch.ones(batch_size, 1).to(device)
    fake_labels = torch.zeros(batch_size, 1).to(device)


    outputs = D_model(images)
    # Loss for real images

    real_score = outputs

    # Loss for fake images
    z = torch.randn(batch_size, latent_size).to(device)
    fake_images = G_model(z)
    outputs = D_model(fake_images)

    fake_score = outputs

    # Sum losses

    # Reset gradients

    # Compute gradients

    # Adjust the parameters using backprop

    return d_loss, real_score, fake_score

Generator Training

In [ ]:
def train_generator(G_model):
    # Generate fake images and calculate loss
    # z = torch.randn(batch_size, latent_size).to(device)
    z = torch.Tensor(np.random.normal(0, 1, (batch_size, latent_size))).to(device)
    fake_images = G_model(z)
    labels = torch.ones(batch_size, 1).to(device)
    # calculate the generator loss

    # Reset gradients

    # Backprop and optimize

    return g_loss, fake_images

### Start the training proccess

In [ ]:
# function to save the sample output of generator
def save_fake_images_mnist(G_model, index):
    # sample_vectors = torch.randn(batch_size, latent_size).to(device)
    sample_vectors = torch.Tensor(np.random.normal(0, 1, (batch_size, latent_size))).to(
        device
    )
    fake_images = G_model(sample_vectors)
    fake_images = fake_images.reshape(fake_images.size(0), 1, 28, 28)
    fake_fname = "fake_images-{0:0=4d}.png".format(index)
    print("Saving", fake_fname)
    save_image(denorm(fake_images), os.path.join(sample_dir, fake_fname), nrow=10)

In [ ]:
%%time
total_step = len(data_loader)
d_losses, g_losses, real_scores, fake_scores = [], [], [], []
G.to(device)
D.to(device)
for epoch in range(num_epochs):
    for i, (images, _) in enumerate(data_loader):
        # Load a batch & transform to vectors
        images = images.reshape(batch_size, -1).to(device)

        # Train the discriminator

        # Train the generator

        # Inspect the losses
        if (i + 1) % 200 == 0:
            d_losses.append(d_loss.item())
            g_losses.append(g_loss.item())
            real_scores.append(real_score.mean().item())
            fake_scores.append(fake_score.mean().item())
            print(
                "Epoch [{}/{}], Step [{}/{}], d_loss: {:.4f}, g_loss: {:.4f}, D(x): {:.2f}, D(G(z)): {:.2f}".format(
                    epoch,
                    num_epochs,
                    i + 1,
                    total_step,
                    d_loss.item(),
                    g_loss.item(),
                    real_score.mean().item(),
                    fake_score.mean().item(),
                )
            )
    # Sample and save images
    save_fake_images_mnist(epoch + 1)

In [ ]:
# show output after epochs training
Image(os.path.join(sample_dir, "fake_images-0225.png"))

In [ ]:
# show the discrimator loss and generator loss
plt.plot(d_losses, "-")
plt.plot(g_losses, "-")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend(["Discriminator", "Generator"])
plt.title("Losses");

In [ ]:
# plot the accuracy of discrimator
plt.plot(real_scores, "-")
plt.plot(fake_scores, "-")
plt.xlabel("epoch")
plt.ylabel("score")
plt.legend(["Real Score", "Fake score"])
plt.title("Scores");

## 2. DCGAN

*Source: <a href="https://arxiv.org/pdf/1511.06434v2.pdf" target="_blank">Radford, Alec, Luke Metz, and Soumith Chintala. "Unsupervised representation learning with deep convolutional generative adversarial networks." arXiv preprint arXiv:1511.06434 (2015)</a>.*

The goal of this part is to enhance your results achieved in the part one with DCGAN. Feel free to reuse your code, improve architecture or training scheme, etc.

In [ ]:
class CFG:
    batch_size = 128
    num_epochs = 300
    workers = 4
    seed = 2021
    image_size = 64
    download = True
    dataroot = "data"
    nc = 3  ## number of chanels
    ngf = 64  # Size of feature maps in generator
    nz = 100  # latent random input vector
    ndf = 64  # Size of feature maps in discriminator
    lr = 0.0002
    device = "cuda" if torch.cuda.is_available() else "cpu"
    sample_dir = "./images/"


if not os.path.exists(CFG.sample_dir):
    os.makedirs(CFG.sample_dir)

cifar_dataset = CIFAR10(
    root=CFG.dataroot,
    download=CFG.download,
    transform=transforms.Compose(
        [
            transforms.Resize([CFG.image_size, CFG.image_size]),
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
        ]
    ),
)

In [ ]:
img, label = cifar_dataset[0]
print("Label: ", label)
plt.hist(img[:, 5:15, 5:15].flatten(), bins=15)

In [ ]:
# unnormalization image from range (-1)-1 to range 0-1 to display it
def denorm(img):
    # write your code

# show  image sample with matplotlib


In [ ]:
# create the dataloader

### DCGAN implementation

In [ ]:
class DCGenerator(nn.Module):
    def __init__(self, nc, nz, ngf):
        # ConvTranspose2d - BatchNorm - Relu -ConvTranspose2d - BatchNorm - Relu -ConvTranspose2d - BatchNorm - Relu
        # ConvTranspose2d - BatchNorm - Relu - ConvTranspose2d - Tanh
        super(Generator, self).__init__()
        ##

    def forward(self, x):
        ##
        return F.tanh(x)


class DCDiscriminator(nn.Module):
    def __init__(self, nc, ndf):
        # conv2d - leaky - conv2d - batchnorm - leaky - conv2d - batchnorm - leaky - conv - batchnorm - leaky - conv2d
        super(Discriminator, self).__init__()
        ##

    def forward(self, x):
        ##
        return F.sigmoid(x)

In [ ]:
# create new Generator model

# DCG = DCGenerator...

# create new Discriminator model

# DCD = DCDiscriminator...

In [ ]:
# show the output of model
y = DCG(torch.randn(2, CFG.nz))

In [ ]:
# define the criterion is nn.BCELoss()
criterion = nn.BCELoss()
## Define the optimizer for generator and discrimator

In [ ]:
# feel free to reuse your training functions

### Start the training proccess

In [ ]:
def save_fake_images_cifar(DCG_model, index):
    sample_vectors = torch.randn(CFG.batch_size, CFG.nz, 1, 1).to(CFG.device)
    fake_images = DCG_model(sample_vectors)
    fake_images = fake_images.reshape(fake_images.size(0), 3, 64, 64)
    fake_fname = "fake_images-{0:0=4d}.png".format(index)
    print("Saving", fake_fname)
    save_image(
        denorm(fake_images), os.path.join(CFG.sample_dir, fake_fname), nrow=10
    )

In [ ]:
%%time
device = CFG.device
num_epochs = CFG.num_epochs
batch_size = CFG.batch_size

total_step = len(data_loader)
d_losses, g_losses, real_scores, fake_scores = [], [], [], []
DCG.to(device)
DCD.to(device)

for epoch in range(num_epochs):
    for i, (images, _) in enumerate(data_loader):
        # Load a batch & transform to vectors
        images = images.reshape(batch_size, -1).to(device)

        # Train the discriminator

        # Train the generator

        # Inspect the losses
        if (i + 1) % 200 == 0:
            d_losses.append(d_loss.item())
            g_losses.append(g_loss.item())
            real_scores.append(real_score.mean().item())
            fake_scores.append(fake_score.mean().item())
            print(
                "Epoch [{}/{}], Step [{}/{}], d_loss: {:.4f}, g_loss: {:.4f}, D(x): {:.2f}, D(G(z)): {:.2f}".format(
                    epoch,
                    num_epochs,
                    i + 1,
                    total_step,
                    d_loss.item(),
                    g_loss.item(),
                    real_score.mean().item(),
                    fake_score.mean().item(),
                )
            )
    # Sample and save images
    save_fake_images_cifar(epoch + 1)

In [ ]:
# show your results!

## 3. Conditional DCGAN

Train a class-conditional DCGAN that generates 64 × 64 CIFAR-10 images for a chosen class. Keep the convolutional DCGAN structure, but pass the class information through feature modulation rather than concatenating a one-hot vector to the noise or image.

Use the activation modulation from Lecture 3. For a class embedding c(y), learn per-channel scale and shift for each feature block:

$$\widetilde h = (1 + \gamma(c(y))) \odot \operatorname{GroupNorm}(h) + \beta(c(y)).$$

Broadcast gamma and beta across the spatial dimensions. Initialize their output projections to zero so that each modulation block starts as an ordinary normalized block. Condition both the generator and the discriminator on the class label.

### Data and configuration

Use the same CIFAR-10 preprocessing as in Part 2: resize RGB images to 64 × 64 and normalize each channel to [-1, 1]. The CIFAR-10 dataset returns both images and integer class labels from 0 to 9.

In [ ]:
import os

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import CIFAR10
from torchvision.utils import save_image


class ConditionalCFG:
    batch_size = 128
    num_epochs = 300
    workers = 4
    image_size = 64
    num_classes = 10
    nc = 3
    ngf = 64
    ndf = 64
    nz = 100
    class_embedding_dim = 128
    lr = 0.0002
    beta1 = 0.5
    beta2 = 0.999
    data_root = "data"
    sample_dir = "./images/conditional_dcgan"
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


os.makedirs(ConditionalCFG.sample_dir, exist_ok=True)

conditional_dataset = CIFAR10(
    root=ConditionalCFG.data_root,
    train=True,
    download=True,
    transform=transforms.Compose(
        [
            transforms.Resize((ConditionalCFG.image_size, ConditionalCFG.image_size)),
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
        ]
    ),
)
conditional_loader = DataLoader(
    conditional_dataset,
    batch_size=ConditionalCFG.batch_size,
    shuffle=True,
    num_workers=ConditionalCFG.workers,
    pin_memory=ConditionalCFG.device.type == "cuda",
)
print("Classes:", conditional_dataset.classes)

### Class modulation

Implement one reusable modulation layer. Its input is a feature map h of shape (B, C, H, W) and a class vector c(y) of shape (B, embedding_dim). The linear projection must produce 2C values, split into gamma and beta and reshape both to (B, C, 1, 1). Use GroupNorm without its own affine parameters.

In [ ]:
class ClassModulation(nn.Module):
    def __init__(self, channels, embedding_dim, num_groups=32):
        super().__init__()
        assert channels % num_groups == 0
        self.norm = nn.GroupNorm(
            num_groups=num_groups, num_channels=channels, affine=False
        )
        self.to_scale_shift = nn.Linear(embedding_dim, 2 * channels)
        nn.init.zeros_(self.to_scale_shift.weight)
        nn.init.zeros_(self.to_scale_shift.bias)

    def forward(self, h, class_vector):
        # TODO: Project class_vector and split it into gamma and beta.
        # TODO: Reshape gamma and beta to (B, C, 1, 1).
        # TODO: Return (1 + gamma) * self.norm(h) + beta.
        raise NotImplementedError("Implement class-conditioned feature modulation")

### Conditional generator and discriminator

The generator receives z with shape (B, nz, 1, 1) and class_ids with shape (B,). It should produce (B, 3, 64, 64) images in [-1, 1]. Use transposed convolutions to grow spatial dimensions 1 → 4 → 8 → 16 → 32 → 64. Apply class modulation and ReLU after each intermediate transposed convolution, then Tanh after the final RGB layer.

The discriminator receives an image and its class_ids. Downsample 64 → 32 → 16 → 8 → 4 → 1, applying class modulation in its hidden blocks and LeakyReLU activations. Return one raw logit per image; the loss below applies BCEWithLogitsLoss. Use the real CIFAR-10 labels for real images and the sampled target labels for generated images.

In [ ]:
class ConditionalDCGenerator(nn.Module):
    def __init__(self, nz, ngf, nc, num_classes, embedding_dim):
        super().__init__()
        self.class_embedding = nn.Embedding(num_classes, embedding_dim)
        self.up1 = nn.ConvTranspose2d(nz, ngf * 8, 4, 1, 0, bias=False)
        self.mod1 = ClassModulation(ngf * 8, embedding_dim)
        self.up2 = nn.ConvTranspose2d(ngf * 8, ngf * 4, 4, 2, 1, bias=False)
        self.mod2 = ClassModulation(ngf * 4, embedding_dim)
        self.up3 = nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False)
        self.mod3 = ClassModulation(ngf * 2, embedding_dim)
        self.up4 = nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False)
        self.mod4 = ClassModulation(ngf, embedding_dim)
        self.to_rgb = nn.ConvTranspose2d(ngf, nc, 4, 2, 1, bias=False)

    def forward(self, z, class_ids):
        # TODO: Embed class_ids once, then reuse the vector in every mod block.
        # TODO: Apply up1..up4, followed by modulation and ReLU.
        # TODO: Apply to_rgb and Tanh. Return images of shape (B, 3, 64, 64).
        raise NotImplementedError("Implement the conditional generator")


class ConditionalDCDiscriminator(nn.Module):
    def __init__(self, nc, ndf, num_classes, embedding_dim):
        super().__init__()
        self.class_embedding = nn.Embedding(num_classes, embedding_dim)
        self.from_rgb = nn.Conv2d(nc, ndf, 4, 2, 1, bias=False)
        self.down2 = nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False)
        self.mod2 = ClassModulation(ndf * 2, embedding_dim)
        self.down3 = nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False)
        self.mod3 = ClassModulation(ndf * 4, embedding_dim)
        self.down4 = nn.Conv2d(ndf * 4, ndf * 8, 4, 2, 1, bias=False)
        self.mod4 = ClassModulation(ndf * 8, embedding_dim)
        self.to_logit = nn.Conv2d(ndf * 8, 1, 4, 1, 0, bias=False)

    def forward(self, images, class_ids):
        # TODO: Embed class_ids once and modulate the hidden feature maps.
        # TODO: Apply LeakyReLU after from_rgb and each modulated block.
        # TODO: Apply to_logit and return raw logits of shape (B,).
        raise NotImplementedError("Implement the conditional discriminator")

### Shape check

Complete the three forward methods above, then run this cell before training. The output dimensions must match the CIFAR-10 images and the discriminator loss.

In [ ]:
conditional_G = ConditionalDCGenerator(
    ConditionalCFG.nz,
    ConditionalCFG.ngf,
    ConditionalCFG.nc,
    ConditionalCFG.num_classes,
    ConditionalCFG.class_embedding_dim,
).to(ConditionalCFG.device)

conditional_D = ConditionalDCDiscriminator(
    ConditionalCFG.nc,
    ConditionalCFG.ndf,
    ConditionalCFG.num_classes,
    ConditionalCFG.class_embedding_dim,
).to(ConditionalCFG.device)

test_z = torch.randn(4, ConditionalCFG.nz, 1, 1, device=ConditionalCFG.device)
test_labels = torch.arange(4, device=ConditionalCFG.device)

with torch.no_grad():
    test_images = conditional_G(test_z, test_labels)
    test_logits = conditional_D(test_images, test_labels)

assert test_images.shape == (4, 3, 64, 64)
assert test_logits.shape == (4,)
print("Generator:", tuple(test_images.shape), "Discriminator:", tuple(test_logits.shape))

### Train the conditional DCGAN

Train D on real pairs (real_image, real_class_id) and fake pairs (G(z, fake_class_id), fake_class_id). Train G so that D accepts its generated pair. Sample the fake class IDs independently and keep each generated image paired with the same ID in G and D. Use the actual batch size, including the final, possibly smaller batch. Do not flatten the images.

In [ ]:
conditional_criterion = nn.BCEWithLogitsLoss()
conditional_g_optimizer = torch.optim.Adam(
    conditional_G.parameters(),
    lr=ConditionalCFG.lr,
    betas=(ConditionalCFG.beta1, ConditionalCFG.beta2),
)
conditional_d_optimizer = torch.optim.Adam(
    conditional_D.parameters(),
    lr=ConditionalCFG.lr,
    betas=(ConditionalCFG.beta1, ConditionalCFG.beta2),
)


def train_conditional_discriminator(real_images, real_class_ids):
    batch_size = real_images.size(0)
    fake_class_ids = torch.randint(
        ConditionalCFG.num_classes, (batch_size,), device=ConditionalCFG.device
    )
    z = torch.randn(batch_size, ConditionalCFG.nz, 1, 1, device=ConditionalCFG.device)
    conditional_d_optimizer.zero_grad(set_to_none=True)

    with torch.no_grad():
        fake_images = conditional_G(z, fake_class_ids)

    real_logits = conditional_D(real_images, real_class_ids)
    fake_logits = conditional_D(fake_images, fake_class_ids)

    # TODO: Compare real_logits with ones and fake_logits with zeros.
    # TODO: Add the two losses, call backward(), step D, and return d_loss.
    raise NotImplementedError("Implement the discriminator update")


def train_conditional_generator(batch_size):
    fake_class_ids = torch.randint(
        ConditionalCFG.num_classes, (batch_size,), device=ConditionalCFG.device
    )
    z = torch.randn(batch_size, ConditionalCFG.nz, 1, 1, device=ConditionalCFG.device)
    conditional_g_optimizer.zero_grad(set_to_none=True)

    # TODO: Generate images with fake_class_ids and score the same pairs with D.
    # TODO: Compare the logits with ones, call backward(), step G, and return g_loss.
    raise NotImplementedError("Implement the generator update")

### Sample and inspect results

The fixed noise below is reused across classes: each row of the saved grid is one CIFAR-10 class, and each column uses the same z across all rows. Inspect whether changing only the class changes the generated object while keeping varied images within each class.

In [ ]:
fixed_noise = torch.randn(
    10, ConditionalCFG.nz, 1, 1, device=ConditionalCFG.device
).repeat(ConditionalCFG.num_classes, 1, 1, 1)
fixed_class_ids = torch.arange(
    ConditionalCFG.num_classes, device=ConditionalCFG.device
).repeat_interleave(10)


@torch.no_grad()
def save_conditional_grid(epoch):
    was_training = conditional_G.training
    conditional_G.eval()
    images = conditional_G(fixed_noise, fixed_class_ids)
    images = (images.clamp(-1, 1) + 1) / 2
    path = os.path.join(ConditionalCFG.sample_dir, f"classes_epoch_{epoch:04d}.png")
    save_image(images, path, nrow=10)
    if was_training:
        conditional_G.train()
    return path

In [ ]:
conditional_d_losses, conditional_g_losses = [], []

for epoch in range(ConditionalCFG.num_epochs):
    for real_images, real_class_ids in conditional_loader:
        real_images = real_images.to(ConditionalCFG.device)
        real_class_ids = real_class_ids.to(ConditionalCFG.device)

        d_loss = train_conditional_discriminator(real_images, real_class_ids)
        g_loss = train_conditional_generator(real_images.size(0))

    conditional_d_losses.append(d_loss.item())
    conditional_g_losses.append(g_loss.item())
    print(
        f"Epoch [{epoch + 1}/{ConditionalCFG.num_epochs}] "
        f"D loss: {d_loss.item():.4f}, G loss: {g_loss.item():.4f}"
    )
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print("Saved:", save_conditional_grid(epoch + 1))

### Your results

Display at least one class grid and the two loss curves. Briefly discuss class consistency, variation within each class, and any training instability. What happens to the generated images if you keep z fixed and change only the class ID?

In [ ]:
# TODO: Display a saved class grid and plot conditional_d_losses / conditional_g_losses.
# TODO: Write a short interpretation of the results.